# News Daily Features

Objetivo: transformar o dataset de notícias em granularidade intraday (`1 linha = 1 notícia`) em um dataset diário (`1 linha = 1 dia`).

### Entrada

`data/processed/energy_br.csv`

### Saída

`data/features/energy_news_daily.csv`

### Features iniciais

- `news_count`
- `signal`
- `avg_score`
- `avg_relevance`
- `positive_count`
- `neutral_count`
- `negative_count`
- `positive_share`
- `neutral_share`
- `negative_share`
- `max_relevance`
- `score_std`
- `signal_ma_3`
- `signal_ma_5`
- `signal_ma_7`

O `signal` diário segue a lógica:

```text
Σ(score × relevance)
────────────────────
    Σ(relevance)
```


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NEWS_PATH = PROJECT_ROOT / "data" / "processed" / "energy_br.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "features" / "energy_news_daily.csv"

print("News input :", NEWS_PATH)
print("Output     :", OUTPUT_PATH)


News input : c:\Users\e702746\OneDrive - EDP\Desktop\projetos\can_ai_listen_to_the_market\data\processed\energy_br.csv
Output     : c:\Users\e702746\OneDrive - EDP\Desktop\projetos\can_ai_listen_to_the_market\data\features\energy_news_daily.csv


## 1. Carregar as notícias


In [2]:
news = pd.read_csv(NEWS_PATH)

news["published_at"] = pd.to_datetime(
    news["published_at"],
    utc=True,
    errors="coerce",
)

news["score"] = pd.to_numeric(
    news["score"],
    errors="coerce",
)

news["relevance"] = pd.to_numeric(
    news["relevance"],
    errors="coerce",
)

print(f"Rows: {len(news):,}")
print(f"Period: {news['published_at'].min()} -> {news['published_at'].max()}")

news.head()


Rows: 2,352
Period: 2026-01-05 11:46:41+00:00 -> 2026-08-24 19:58:39+00:00


,source,title,summary,url,published_at,fetched_at,category,sentiment,score,relevance,reason
0,megawhat,Clearing pode destravar liquidez e reduzir ris...,Sinal de preços no mercado livre A criação de ...,https://megawhat.uol.com.br/economia-e-politic...,2026-08-20 18:23:48+00:00,2026-08-20T19:26:41+00:00,energy_br,positive,0.46,0.82,A criação de uma clearing tende a reduzir risc...
1,megawhat,Data centers de 2 GW podem dar ‘tranco’ no sis...,Diretor de Planejamento do Operador Nacional d...,https://megawhat.uol.com.br/podcasts/minutomeg...,2026-08-20 18:09:01+00:00,2026-08-20T19:26:41+00:00,energy_br,neutral,0.10,0.78,A notícia sugere aumento potencial de demanda ...
2,megawhat,Petrobras amplia contratos de acesso de tercei...,Instalações de gás da Petrobras / crédito: Pet...,https://megawhat.uol.com.br/economia-e-politic...,2026-08-20 16:01:30+00:00,2026-08-20T19:26:41+00:00,energy_br,positive,0.57,0.86,A ampliação de acesso de terceiros à infraestr...
3,megawhat,"Termelétrica GNA II, de 1,7 GW de capacidade, ...","Com entrada em operação da GNA II, complexo no...",https://megawhat.uol.com.br/economia-e-politic...,2026-08-20 15:47:39+00:00,2026-08-20T19:26:41+00:00,energy_br,negative,-0.62,0.79,A interrupção de uma usina relevante reduz tem...
4,megawhat,Terminais de GNL já têm concorrência e não pre...,"Lino Cançado, CEO da Eneva, no MinutoMega Talk...",https://megawhat.uol.com.br/podcasts/minutomeg...,2026-08-20 14:59:10+00:00,2026-08-20T19:26:41+00:00,energy_br,neutral,0.10,0.78,A notícia trata de possível regulação de acess...


## 2. Filtrar notícias analisadas

Para gerar features de sentimento, consideramos somente notícias com `score` e `relevance` disponíveis.

In [3]:
analyzed = news.dropna(
    subset=[
        "published_at",
        "score",
        "relevance",
    ]
).copy()

analyzed["date"] = analyzed["published_at"].dt.date

print(f"Analyzed news: {len(analyzed):,}")
print(f"Unique days  : {analyzed['date'].nunique():,}")

analyzed[
    [
        "date",
        "source",
        "title",
        "sentiment",
        "score",
        "relevance",
    ]
].head()


Analyzed news: 2,352
Unique days  : 161


,date,source,title,sentiment,score,relevance
0,2026-08-20,megawhat,Clearing pode destravar liquidez e reduzir ris...,positive,0.46,0.82
1,2026-08-20,megawhat,Data centers de 2 GW podem dar ‘tranco’ no sis...,neutral,0.10,0.78
2,2026-08-20,megawhat,Petrobras amplia contratos de acesso de tercei...,positive,0.57,0.86
3,2026-08-20,megawhat,"Termelétrica GNA II, de 1,7 GW de capacidade, ...",negative,-0.62,0.79
4,2026-08-20,megawhat,Terminais de GNL já têm concorrência e não pre...,neutral,0.10,0.78


## 3. Contribuição ponderada por notícia

Cada notícia contribui para o sinal diário por:

`weighted_score = score × relevance`

A relevância funciona como peso na agregação.

In [4]:
analyzed["weighted_score"] = (
    analyzed["score"]
    * analyzed["relevance"]
)

analyzed[
    [
        "date",
        "title",
        "score",
        "relevance",
        "weighted_score",
    ]
].head()


,date,title,score,relevance,weighted_score
0,2026-08-20,Clearing pode destravar liquidez e reduzir ris...,0.46,0.82,0.3772
1,2026-08-20,Data centers de 2 GW podem dar ‘tranco’ no sis...,0.10,0.78,0.0780
2,2026-08-20,Petrobras amplia contratos de acesso de tercei...,0.57,0.86,0.4902
3,2026-08-20,"Termelétrica GNA II, de 1,7 GW de capacidade, ...",-0.62,0.79,-0.4898
4,2026-08-20,Terminais de GNL já têm concorrência e não pre...,0.10,0.78,0.0780


## 4. Features diárias

Agregamos as notícias por `published_at`.

A granularidade final passa a ser diária


In [5]:
sentiment_daily = (
    analyzed
    .assign(
        sentiment=analyzed["sentiment"]
        .astype(str)
        .str.lower()
    )
    .groupby(
        ["date", "sentiment"]
    )
    .size()
    .unstack(fill_value=0)
)

for column in [
    "positive",
    "neutral",
    "negative",
]:
    if column not in sentiment_daily.columns:
        sentiment_daily[column] = 0

sentiment_daily = sentiment_daily[
    [
        "positive",
        "neutral",
        "negative",
    ]
].rename(
    columns={
        "positive": "positive_count",
        "neutral": "neutral_count",
        "negative": "negative_count",
    }
)

sentiment_daily.head()


sentiment,positive_count,neutral_count,negative_count
date,,,
2026-01-05,0,12,2
2026-01-06,0,12,1
2026-01-07,0,14,0
2026-01-08,1,11,0
2026-01-09,0,12,0


In [7]:
daily = (
    analyzed
    .groupby("date")
    .agg(
        news_count=("score", "count"),
        weighted_sum=("weighted_score", "sum"),
        total_relevance=("relevance", "sum"),
        avg_score=("score", "mean"),
        avg_relevance=("relevance", "mean"),
        max_relevance=("relevance", "max"),
        score_std=("score", "std"),
    )
    .join(sentiment_daily)
    .reset_index()
)

daily["signal"] = np.where(
    daily["total_relevance"] > 0,
    daily["weighted_sum"] / daily["total_relevance"],
    0.0,
)

daily["positive_share"] = (
    daily["positive_count"] / daily["news_count"]
)

daily["neutral_share"] = (
    daily["neutral_count"] / daily["news_count"]
)

daily["negative_share"] = (
    daily["negative_count"] / daily["news_count"]
)

daily["score_std"] = daily["score_std"].fillna(0.0)

daily.head()


,date,news_count,weighted_sum,total_relevance,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,signal,positive_share,neutral_share,negative_share
0,2026-01-05,14,-0.3008,2.99,-0.037857,0.213571,0.63,0.109064,0,12,2,-0.100602,0.000000,0.857143,0.142857
1,2026-01-06,13,-0.1240,1.82,-0.011538,0.140000,0.55,0.074032,0,12,1,-0.068132,0.000000,0.923077,0.076923
2,2026-01-07,14,0.0534,1.78,0.022857,0.127143,0.26,0.024939,0,14,0,0.030000,0.000000,1.000000,0.000000
3,2026-01-08,12,0.1886,1.76,0.031667,0.146667,0.62,0.080547,1,11,0,0.107159,0.083333,0.916667,0.000000
4,2026-01-09,12,0.0410,1.46,0.012500,0.121667,0.32,0.031079,0,12,0,0.028082,0.000000,1.000000,0.000000


## 5. Médias móveis

As médias móveis ajudam a reduzir ruído diário e capturar persistência do sinal.

Começamos com janelas simples de 3, 5 e 7 dias de observação.


In [8]:
daily = daily.sort_values("date").reset_index(drop=True)

for window in [3, 5, 7]:
    daily[f"signal_ma_{window}"] = (
        daily["signal"]
        .rolling(
            window=window,
            min_periods=1,
        )
        .mean()
    )

daily[
    [
        "date",
        "signal",
        "signal_ma_3",
        "signal_ma_5",
        "signal_ma_7",
    ]
].tail(10)


,date,signal,signal_ma_3,signal_ma_5,signal_ma_7
151,2026-08-11,-0.055944,-0.238559,-0.191674,-0.154486
152,2026-08-12,0.078433,-0.062415,-0.159553,-0.126437
153,2026-08-13,-0.093125,-0.023545,-0.146074,-0.139009
154,2026-08-14,-0.120915,-0.045202,-0.080257,-0.144543
155,2026-08-17,0.055164,-0.052959,-0.027277,-0.113731
156,2026-08-18,0.033333,-0.010806,-0.009422,-0.044684
157,2026-08-19,0.216449,0.101649,0.018181,0.016199
158,2026-08-20,0.231530,0.160437,0.083112,0.057267
159,2026-08-21,-0.177526,0.090151,0.071790,0.020701
160,2026-08-24,-0.065420,-0.003805,0.047673,0.024659


## 6. Dataset final

In [9]:
feature_columns = [
    "date",
    "news_count",
    "signal",
    "avg_score",
    "avg_relevance",
    "max_relevance",
    "score_std",
    "positive_count",
    "neutral_count",
    "negative_count",
    "positive_share",
    "neutral_share",
    "negative_share",
    "signal_ma_3",
    "signal_ma_5",
    "signal_ma_7",
]

daily_features = daily[feature_columns].copy()

numeric_columns = daily_features.select_dtypes(
    include="number"
).columns

daily_features[numeric_columns] = (
    daily_features[numeric_columns]
    .round(4)
)

daily_features.tail(10)


,date,news_count,signal,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,positive_share,neutral_share,negative_share,signal_ma_3,signal_ma_5,signal_ma_7
151,2026-08-11,20,-0.0559,-0.0050,0.1430,0.65,0.0742,0,19,1,0.0000,0.9500,0.0500,-0.2386,-0.1917,-0.1545
152,2026-08-12,13,0.0784,0.0285,0.1669,0.66,0.0615,1,12,0,0.0769,0.9231,0.0000,-0.0624,-0.1596,-0.1264
153,2026-08-13,12,-0.0931,-0.0267,0.2133,0.74,0.0887,0,10,2,0.0000,0.8333,0.1667,-0.0235,-0.1461,-0.1390
154,2026-08-14,19,-0.1209,-0.0147,0.1495,0.68,0.1419,1,17,1,0.0526,0.8947,0.0526,-0.0452,-0.0803,-0.1445
155,2026-08-17,18,0.0552,0.0250,0.1356,0.45,0.0503,1,17,0,0.0556,0.9444,0.0000,-0.0530,-0.0273,-0.1137
156,2026-08-18,11,0.0333,0.0182,0.1036,0.27,0.0337,0,11,0,0.0000,1.0000,0.0000,-0.0108,-0.0094,-0.0447
157,2026-08-19,13,0.2164,0.0485,0.2123,0.92,0.2064,1,11,1,0.0769,0.8462,0.0769,0.1016,0.0182,0.0162
158,2026-08-20,15,0.2315,0.2093,0.7060,0.89,0.3793,8,5,2,0.5333,0.3333,0.1333,0.1604,0.0831,0.0573
159,2026-08-21,13,-0.1775,-0.1123,0.5877,0.92,0.4403,2,6,5,0.1538,0.4615,0.3846,0.0902,0.0718,0.0207
160,2026-08-24,22,-0.0654,-0.0364,0.7473,0.95,0.4553,5,8,9,0.2273,0.3636,0.4091,-0.0038,0.0477,0.0247


## 7. Validações rápidas

- uma linha por dia;
- `signal` entre -1 e +1;
- shares entre 0 e 1;
- contagem de sentimentos igual a `news_count`.


In [11]:
assert daily_features["date"].is_unique

assert daily_features["signal"].between(
    -1,
    1,
).all()

for column in [
    "positive_share",
    "neutral_share",
    "negative_share",
]:
    assert daily_features[column].between(
        0,
        1,
    ).all()

sentiment_total = (
    daily_features["positive_count"]
    + daily_features["neutral_count"]
    + daily_features["negative_count"]
)

assert (
    sentiment_total
    == daily_features["news_count"]
).all()

print("✓ Validations passed")


✓ Validations passed


## 8. Persistir features diárias

`02_market_news_eda.ipynb`


In [12]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

daily_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(
    f"✓ Saved {len(daily_features):,} daily rows "
    f"to {OUTPUT_PATH}"
)


✓ Saved 161 daily rows to c:\Users\e702746\OneDrive - EDP\Desktop\projetos\can_ai_listen_to_the_market\data\features\energy_news_daily.csv


## 9. Resumo

O dataset final contém uma linha por dia com:

- intensidade e direção agregada das notícias;
- relevância média;
- volume de notícias;
- distribuição de sentimento;
- dispersão dos scores;
- médias móveis do sinal.


In [13]:
daily_features.head()

,date,news_count,signal,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,positive_share,neutral_share,negative_share,signal_ma_3,signal_ma_5,signal_ma_7
0,2026-01-05,14,-0.1006,-0.0379,0.2136,0.63,0.1091,0,12,2,0.0000,0.8571,0.1429,-0.1006,-0.1006,-0.1006
1,2026-01-06,13,-0.0681,-0.0115,0.1400,0.55,0.0740,0,12,1,0.0000,0.9231,0.0769,-0.0844,-0.0844,-0.0844
2,2026-01-07,14,0.0300,0.0229,0.1271,0.26,0.0249,0,14,0,0.0000,1.0000,0.0000,-0.0462,-0.0462,-0.0462
3,2026-01-08,12,0.1072,0.0317,0.1467,0.62,0.0805,1,11,0,0.0833,0.9167,0.0000,0.0230,-0.0079,-0.0079
4,2026-01-09,12,0.0281,0.0125,0.1217,0.32,0.0311,0,12,0,0.0000,1.0000,0.0000,0.0551,-0.0007,-0.0007
